# Question 2 - Feed-forward SNN on Poisson-encoded MNIST (Brian2)

**Pipeline.** 28x28 grey image -> Poisson spike trains (`rate = intensity x 100 Hz`, encoding window `T_enc = 100 ms`)
-> hidden layer of LIF neurons with fixed random excitatory weights -> spike counts per hidden neuron
-> linear read-out (`sklearn.LogisticRegression`) trained offline on the spike-count vectors.

**Neuron model.** Standard LIF with membrane time constant `tau_m`, `V_rest = -70 mV`, `V_reset = -70 mV`,
`V_th = -50 mV`, absolute refractory period `2 ms`:

$$\tau_m \frac{dv}{dt} = (V_{rest} - v), \qquad v \leftarrow v + w_{ij} \text{ on each input spike}, \qquad v > V_{th} \Rightarrow \text{spike},\; v \leftarrow V_{reset}.$$

**Synapses.** Instantaneous (delta) current-based EPSPs. Weights `w ~ N(0.5, 0.1)` clipped to be non-negative (excitatory only),
multiplied by a scale `w_scale` (mV) and a 0/1 connectivity mask with connection probability `p_conn`.

**Implementation trick.** To make Brian2 fast enough with the pure-numpy backend, `B = 25` independent copies of the
network are simulated in one `Network` (block-diagonal synapse matrix). The Poisson rates of the whole dataset are stored in a
2-D `TimedArray` (`dt = T_enc`) so a *single* `run()` streams all images through the network. At every image boundary the membrane
potential is reset (`run_regularly`), so images are independent. Hidden spikes are binned per (image, neuron, 10-ms bin), which
lets us study accuracy as a function of the decision window.

Experiments in this notebook

1. **Baseline** - accuracy / F1, confusion matrix, firing-rate statistics, rasters + voltage traces, accuracy vs. decision window.
2. **Tweak 1 (main): membrane time constant sweep** `tau_m in {5, 10, 20, 50} ms` - with fixed weights and with "drive-matched" weights (`w_scale ~ 1/tau_m`).
3. **Tweak 2: sparse connectivity** `p_conn in {1.0, 0.5, 0.2, 0.1, 0.05, 0.02}` with matched mean drive, plus example filters.
4. **Tweak 3: homeostatic firing-rate regulation** - per-neuron adaptive thresholds driven to a target rate, then frozen.

## 1. Code: data loading, network construction, simulation and read-out

In [ ]:
import os, time, json
import numpy as np
import matplotlib.pyplot as plt
import gzip
import os
import struct
import time

import numpy as np
from brian2 import (Hz, Network, NeuronGroup, PoissonGroup, SpikeMonitor,
                    StateMonitor, Synapses, TimedArray, defaultclock, ms, mV,
                    prefs, second, volt)

prefs.codegen.target = "numpy"      # no compiler needed on Windows

os.makedirs('figures', exist_ok=True)
os.makedirs('results', exist_ok=True)

### Data

In [ ]:
def _read_idx(path):
    with gzip.open(path, "rb") as f:
        magic, = struct.unpack(">I", f.read(4))
        ndim = magic & 0xFF
        shape = struct.unpack(">" + "I" * ndim, f.read(4 * ndim))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(shape)


def load_mnist(root="data"):
    """Returns X_train (60000, 784) in [0,1], y_train, X_test (10000, 784), y_test."""
    Xtr = _read_idx(os.path.join(root, "train-images-idx3-ubyte.gz")).reshape(-1, 784) / 255.0
    ytr = _read_idx(os.path.join(root, "train-labels-idx1-ubyte.gz")).astype(int)
    Xte = _read_idx(os.path.join(root, "t10k-images-idx3-ubyte.gz")).reshape(-1, 784) / 255.0
    yte = _read_idx(os.path.join(root, "t10k-labels-idx1-ubyte.gz")).astype(int)
    return Xtr.astype(np.float64), ytr, Xte.astype(np.float64), yte

### Default parameters

In [ ]:
P = dict(
    n_in=784,
    n_hidden=200,
    t_enc=100 * ms,          # encoding window per image
    max_rate=100 * Hz,       # pixel intensity 1.0 -> 100 Hz
    tau_m=20 * ms,
    v_rest=-70 * mV,
    v_reset=-70 * mV,
    v_th=-50 * mV,
    t_ref=2 * ms,
    w_mean=0.5,              # weights ~ N(0.5, 0.1), clipped >= 0 (dimensionless)
    w_std=0.1,
    w_scale=0.25 * mV,       # EPSP jump per unit weight
    p_conn=1.0,              # connection probability (1.0 = dense)
    batch=25,                # images simulated in parallel
    dt=0.1 * ms,
    n_bins=10,               # time bins per image (for latency analysis)
)


def make_weights(p, seed=0):
    """Random excitatory weight matrix W (n_in x n_hidden) and the 0/1 mask."""
    rng = np.random.default_rng(seed)
    W = rng.normal(p["w_mean"], p["w_std"], size=(p["n_in"], p["n_hidden"]))
    W = np.clip(W, 0.0, None)
    mask = rng.random((p["n_in"], p["n_hidden"])) < p["p_conn"]
    return W * mask, mask

### Network construction

In [ ]:
LIF_EQS = """
dv/dt = (v_rest - v) / tau_m : volt (unless refractory)
theta : volt   (constant)
"""


def build_network(rates_vals, W, p, theta_frozen=None, record_v=None, homeostasis=None):
    """
    rates_vals : (n_chunks, B*n_in) array of Poisson rates (Hz, plain floats)
    W          : (n_in, n_hidden) weights (dimensionless, multiplied by w_scale)
    theta_frozen : optional per-neuron threshold array (volt); default p['v_th']
    homeostasis: None or dict(r_target=..Hz, tau_r=..s, eta=..volt, theta_min, theta_max)
                 -> threshold becomes a dynamic variable driven to a target rate
    """
    B, n_in, n_h = p["batch"], p["n_in"], p["n_hidden"]
    stim = TimedArray(rates_vals * Hz, dt=p["t_enc"])
    inp = PoissonGroup(B * n_in, rates="stim(t, i)", namespace={"stim": stim}, name="inp")

    ns = dict(stim=stim, v_rest=p["v_rest"], tau_m=p["tau_m"])
    if homeostasis is None:
        eqs = LIF_EQS
    else:
        # r      : low-pass estimate of the neuron's own firing rate
        # theta  : adaptive threshold, integrates (r - r_target)
        eqs = """
        dv/dt = (v_rest - v) / tau_m : volt (unless refractory)
        dr/dt = -r / tau_r : Hz
        dtheta/dt = eta * (r - r_target) : volt
        """
        ns.update(tau_r=homeostasis["tau_r"], eta=homeostasis["eta"],
                  r_target=homeostasis["r_target"],
                  theta_min=homeostasis["theta_min"], theta_max=homeostasis["theta_max"])
    hid = NeuronGroup(B * n_h, eqs, threshold="v > theta",
                      reset="v = v_reset" if homeostasis is None else "v = v_reset; r += 1/tau_r",
                      refractory=p["t_ref"], method="euler", namespace={**ns, "v_reset": p["v_reset"]},
                      name="hid")
    hid.v = p["v_rest"]
    hid.theta = p["v_th"] if theta_frozen is None else np.tile(theta_frozen, B)
    # reset membrane at every image boundary so images are independent
    hid.run_regularly("v = v_rest", dt=p["t_enc"], when="start")
    if homeostasis is not None:
        hid.run_regularly("theta = clip(theta, theta_min, theta_max)", dt=1 * ms)

    syn = Synapses(inp, hid, "w : volt", on_pre="v_post += w", name="syn")
    pre, post = np.nonzero(W)
    i_all = np.concatenate([pre + b * n_in for b in range(B)])
    j_all = np.concatenate([post + b * n_h for b in range(B)])
    syn.connect(i=i_all, j=j_all)
    syn.w = np.tile(W[pre, post], B) * p["w_scale"]

    mon = SpikeMonitor(hid, name="mon_hid")
    objs = [inp, hid, syn, mon]
    if record_v is not None:
        objs.append(StateMonitor(hid, "v", record=record_v, name="mon_v"))
        objs.append(SpikeMonitor(inp, name="mon_in"))
    net = Network(objs)
    return net


def simulate(X, W, p, theta_frozen=None, verbose=True):
    """
    Stream all images in X (n_img, 784) through the network.
    Returns counts_t : (n_img, n_hidden, n_bins) spike counts per time bin.
    """
    defaultclock.dt = p["dt"]
    B, n_in, n_h = p["batch"], p["n_in"], p["n_hidden"]
    n_img = X.shape[0]
    n_chunks = int(np.ceil(n_img / B))
    Xpad = np.zeros((n_chunks * B, n_in))
    Xpad[:n_img] = X
    rates_vals = Xpad.reshape(n_chunks, B * n_in) * float(p["max_rate"] / Hz)

    net = build_network(rates_vals, W, p, theta_frozen=theta_frozen)
    t0 = time.time()
    net.run(n_chunks * p["t_enc"])
    wall = time.time() - t0

    mon = net["mon_hid"]
    t = np.asarray(mon.t / p["t_enc"])          # time in units of t_enc
    k = np.floor(t).astype(int)                 # chunk index
    frac = t - k
    tb = np.minimum((frac * p["n_bins"]).astype(int), p["n_bins"] - 1)
    idx = np.asarray(mon.i)
    b = idx // n_h
    h = idx % n_h
    img = k * B + b
    flat = (img * n_h + h) * p["n_bins"] + tb
    counts_t = np.bincount(flat, minlength=n_chunks * B * n_h * p["n_bins"])
    counts_t = counts_t.reshape(n_chunks * B, n_h, p["n_bins"])[:n_img]
    if verbose:
        rate = counts_t.sum() / (n_img * n_h * float(p["t_enc"] / second))
        print(f"  simulated {n_img} images in {wall:6.1f}s  |  mean hidden rate = {rate:5.1f} Hz  "
              f"|  spikes/image = {counts_t.sum() / n_img:7.1f}")
    return counts_t


def calibrate_homeostasis(X_calib, W, p, r_target=20 * Hz, tau_r=0.5 * second,
                          eta=0.5 * mV, theta_min=None, theta_max=None, sim_time=20 * second,
                          verbose=True):
    """
    Run the network with an adaptive threshold on a calibration stream and
    return the per-neuron thresholds reached at the end (then frozen).
    Threshold dynamics:  d theta/dt = eta * (r - r_target), r = low-passed own rate.
    The B parallel replicas each see a different image stream; the calibration
    images are cycled until every replica has adapted for `sim_time` of
    simulated time (>> tau_r, so the thresholds converge).
    Returns the frozen thresholds and a (n_chunks, n_hidden) trace of replica 0.
    """
    defaultclock.dt = p["dt"]
    B, n_in, n_h = p["batch"], p["n_in"], p["n_hidden"]
    n_needed = int(round(float(sim_time / p["t_enc"]))) * B
    reps = int(np.ceil(n_needed / X_calib.shape[0]))
    X_calib = np.tile(X_calib, (reps, 1))[:n_needed]
    n_img = X_calib.shape[0]
    n_chunks = int(np.ceil(n_img / B))
    Xpad = np.zeros((n_chunks * B, n_in))
    Xpad[:n_img] = X_calib
    rates_vals = Xpad.reshape(n_chunks, B * n_in) * float(p["max_rate"] / Hz)
    hp = dict(r_target=r_target, tau_r=tau_r, eta=eta,
              theta_min=p["v_rest"] + 2 * mV if theta_min is None else theta_min,
              theta_max=p["v_th"] + 60 * mV if theta_max is None else theta_max)
    net = build_network(rates_vals, W, p, homeostasis=hp)
    hid = net["hid"]
    hid.r = 0 * Hz
    trace = []
    for c in range(n_chunks):
        net.run(p["t_enc"])
        trace.append(np.asarray(hid.theta[:n_h] / mV).copy())
    theta_B = np.asarray(hid.theta / volt).reshape(B, n_h)
    theta_out = theta_B.mean(axis=0) * volt      # average over the B replicas
    if verbose:
        print(f"  homeostasis: theta mean={np.mean(theta_out/mV):.1f} mV  "
              f"min={np.min(theta_out/mV):.1f}  max={np.max(theta_out/mV):.1f} mV")
    return theta_out, np.array(trace)


def demo_run(X, W, p, theta_frozen=None, record_neurons=(0, 1, 2)):
    """Small run (a few images, batch=1) returning rasters + voltage traces for plots."""
    defaultclock.dt = p["dt"]
    q = dict(p, batch=1)
    n_img = X.shape[0]
    rates_vals = X.reshape(n_img, p["n_in"]) * float(p["max_rate"] / Hz)
    net = build_network(rates_vals, W, q, theta_frozen=theta_frozen, record_v=list(record_neurons))
    net.run(n_img * p["t_enc"])
    out = dict(
        in_t=np.asarray(net["mon_in"].t / ms), in_i=np.asarray(net["mon_in"].i),
        hid_t=np.asarray(net["mon_hid"].t / ms), hid_i=np.asarray(net["mon_hid"].i),
        v_t=np.asarray(net["mon_v"].t / ms), v=np.asarray(net["mon_v"].v / mV),
        v_neurons=list(record_neurons),
    )
    return out

### Read-out

In [ ]:
def train_readout(F_train, y_train, F_test, y_test, C=1.0):
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import accuracy_score, f1_score
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(C=C, max_iter=3000))
    clf.fit(F_train, y_train)
    pred = clf.predict(F_test)
    return dict(acc=accuracy_score(y_test, pred),
                f1=f1_score(y_test, pred, average="macro"),
                train_acc=clf.score(F_train, y_train),
                clf=clf, pred=pred)


def accuracy_vs_window(counts_tr, y_tr, counts_te, y_te, p):
    """Accuracy when the read-out only sees spikes from the first k time bins."""
    n_bins = counts_tr.shape[2]
    bin_ms = float(p["t_enc"] / ms) / n_bins
    windows, accs = [], []
    for k in range(1, n_bins + 1):
        Ftr = counts_tr[:, :, :k].sum(axis=2)
        Fte = counts_te[:, :, :k].sum(axis=2)
        r = train_readout(Ftr, y_tr, Fte, y_te)
        windows.append(k * bin_ms)
        accs.append(r["acc"])
    return np.array(windows), np.array(accs)

### Plot style

A single muted style for all figures (categorical palette: blue, orange, aqua, yellow, magenta, violet).

In [ ]:
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6", "#e34948", "#008300"]
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=PAL), "font.size": 10, "legend.frameon": False, "lines.linewidth": 1.6,
})

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"figures/{name}.png", bbox_inches="tight")
    plt.show()

## 2. Setup: data subset, baseline parameters, random weights

To keep Brian2 run-times reasonable we use the first 5 000 training images (read-out training) and the first 1 000 test images
(evaluation). A separate 500-image slice of the training set is reserved for homeostatic calibration (Tweak 3).

In [ ]:
RNG_SEED = 0
N_TRAIN, N_TEST, N_CALIB = 5000, 1000, 500
C_READOUT = 0.1                       # L2 strength of the logistic-regression read-out (1/C)

from brian2 import seed as brian_seed
brian_seed(RNG_SEED)                 # Poisson spike trains are reproducible
Xtr_all, ytr_all, Xte_all, yte_all = load_mnist("data")
Xtr, ytr = Xtr_all[:N_TRAIN], ytr_all[:N_TRAIN]
Xte, yte = Xte_all[:N_TEST], yte_all[:N_TEST]
X_calib = Xtr_all[N_TRAIN:N_TRAIN + N_CALIB]

BASE = dict(P, n_hidden=300, p_conn=0.2, w_scale=1.25 * mV, tau_m=20 * ms)
W0, mask0 = make_weights(BASE, seed=RNG_SEED)

print("train", Xtr.shape, "test", Xte.shape)
print(f"hidden neurons: {BASE['n_hidden']},  p_conn = {BASE['p_conn']},  synapses: {int(mask0.sum())}, "
      f"fan-in per hidden neuron ~ {mask0.sum(0).mean():.0f}")
print(f"mean input spikes / image = {Xtr.sum(1).mean() * float(BASE['max_rate'] * BASE['t_enc']):.0f}")
RESULTS = {}

## 3. Baseline network

### 3.1 What the network sees: Poisson encoding, hidden spikes and membrane traces

In [ ]:
demo = demo_run(Xte[:3], W0, BASE, record_neurons=(0, 1, 2))
T = float(BASE["t_enc"] / ms)

fig, ax = plt.subplots(2, 2, figsize=(11, 6.5), gridspec_kw=dict(width_ratios=[1, 3.2]))
ax[0, 0].imshow(Xte[0].reshape(28, 28), cmap="gray_r"); ax[0, 0].set_title(f"test image 0 (label {yte[0]})"); ax[0, 0].axis("off")
im = ax[1, 0].imshow(Xte[0].reshape(28, 28) * float(BASE["max_rate"] / Hz), cmap="Blues"); ax[1, 0].axis("off")
ax[1, 0].set_title("Poisson rate map (Hz)"); plt.colorbar(im, ax=ax[1, 0], fraction=0.046)
sel = demo["in_i"] < 784
ax[0, 1].scatter(demo["in_t"], demo["in_i"], s=1, color=PAL[0], alpha=0.6)
ax[0, 1].set_ylabel("input (pixel) index"); ax[0, 1].set_title("Poisson input spikes - 3 consecutive images (100 ms each)")
ax[1, 1].scatter(demo["hid_t"], demo["hid_i"], s=2, color=PAL[1], alpha=0.7)
ax[1, 1].set_ylabel("hidden neuron"); ax[1, 1].set_xlabel("time (ms)"); ax[1, 1].set_title("hidden LIF spikes")
for a in ax[:, 1]:
    for k in range(1, 3): a.axvline(k * T, color="#898781", lw=0.8, ls="--")
    a.set_xlim(0, 3 * T)
savefig("fig_baseline_rasters")

fig, ax = plt.subplots(3, 1, figsize=(11, 5), sharex=True)
for k, n in enumerate(demo["v_neurons"]):
    ax[k].plot(demo["v_t"], demo["v"][k], color=PAL[k], lw=1)
    ax[k].axhline(-50, color="#898781", lw=0.8, ls=":"); ax[k].set_ylabel(f"n{n}  v (mV)")
    sp = demo["hid_t"][demo["hid_i"] == n]
    ax[k].vlines(sp, -50, -40, color=PAL[k], lw=1)
    for j in range(1, 3): ax[k].axvline(j * T, color="#898781", lw=0.8, ls="--")
ax[0].set_title("membrane potential of three hidden neurons (ticks = spikes, dotted = threshold)")
ax[-1].set_xlabel("time (ms)"); ax[-1].set_xlim(0, 3 * T)
savefig("fig_baseline_vtraces")

### 3.2 Run the whole dataset, train the read-out

In [ ]:
t0 = time.time()
print("train set"); counts_tr = simulate(Xtr, W0, BASE)
print("test set");  counts_te = simulate(Xte, W0, BASE)
print(f"total simulation wall-time: {time.time() - t0:.0f} s")

F_tr, F_te = counts_tr.sum(2), counts_te.sum(2)          # spike-count features (n_img x n_hidden)
base = train_readout(F_tr, ytr, F_te, yte, C=C_READOUT)
rate_hz = F_te.mean(0) / float(BASE["t_enc"] / second)   # per-neuron mean rate on the test set

from sklearn.linear_model import LogisticRegression
ref = LogisticRegression(C=C_READOUT, max_iter=3000).fit(Xtr, ytr).score(Xte, yte)

RESULTS["baseline"] = dict(acc=base["acc"], f1=base["f1"], train_acc=base["train_acc"],
                           mean_rate_hz=float(rate_hz.mean()), rate_std_hz=float(rate_hz.std()),
                           spikes_per_image=float(F_te.sum(1).mean()), silent_frac=float((F_tr.sum(0) == 0).mean()),
                           pixel_logreg_acc=float(ref))
print(f"\nBASELINE  test accuracy = {base['acc']:.3f}   macro-F1 = {base['f1']:.3f}   (train acc {base['train_acc']:.3f})")
print(f"mean hidden rate = {rate_hz.mean():.1f} +- {rate_hz.std():.1f} Hz,  hidden spikes/image = {F_te.sum(1).mean():.0f}")
print(f"reference: logistic regression on raw pixels (same 5000 images) = {ref:.3f}")

### 3.3 Baseline diagnostics: confusion matrix, rate distribution, accuracy vs. decision window

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(yte, base["pred"])
win, acc_win = accuracy_vs_window(counts_tr, ytr, counts_te, yte, BASE)
RESULTS["baseline"]["acc_vs_window"] = dict(window_ms=win.tolist(), acc=acc_win.tolist())

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
im = ax[0].imshow(cm, cmap="Blues"); ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true"); ax[0].grid(False)
ax[0].set_title(f"confusion matrix (acc {base['acc']:.3f})"); ax[0].set_xticks(range(10)); ax[0].set_yticks(range(10))
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0: ax[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > cm.max() / 2 else "#0b0b0b")
ax[1].hist(rate_hz, bins=25, color=PAL[0]); ax[1].set_xlabel("mean rate per hidden neuron (Hz)"); ax[1].set_ylabel("# neurons")
ax[1].set_title("hidden firing-rate distribution (test set)")
ax[2].plot(win, acc_win, "o-", color=PAL[1]); ax[2].axhline(ref, color="#898781", ls="--", lw=1)
ax[2].text(win[0], ref + 0.01, "raw-pixel logistic regression", color="#52514e", fontsize=8)
ax[2].set_xlabel("decision window (ms after image onset)"); ax[2].set_ylabel("test accuracy"); ax[2].set_ylim(0, 1)
ax[2].set_title("accuracy vs. read-out window")
savefig("fig_baseline_diag")

### 3.4 Baseline - analysis

*(filled in after the run - see the printed numbers above)*

## 4. Tweak 1 (main): systematically varying the membrane time constant $\tau_m$

$\tau_m$ sets how long an EPSP survives, i.e. the window over which input spikes are integrated. We sweep
$\tau_m \in \{5, 10, 20, 50\}$ ms in two regimes:

* **fixed weights** - everything else as in the baseline. Because the steady-state depolarisation of a LIF neuron under a
  Poisson barrage is $\bar v - V_{rest} \approx \tau_m \sum_i w_i r_i$, a longer $\tau_m$ *also* means a stronger mean drive,
  so firing rates change dramatically;
* **drive-matched weights** - $w_{scale} \propto 1/\tau_m$ so that the mean depolarisation is the same for all $\tau_m$.
  This isolates the effect of the integration window / temporal smoothing from the effect of the mean drive.

For each setting we report accuracy, F1, mean rate, spikes per image, the population PSTH, mean first-spike latency and an example voltage trace.

In [ ]:
TAUS = [5, 10, 20, 50]
res_tau = {"fixed": {}, "matched": {}}
traces_tau = {"fixed": {}, "matched": {}}

def first_spike_latency(d, n_hidden, n_img, T):
    """mean (over images and neurons that fired) time from image onset to the neuron's first spike"""
    lat = []
    for k in range(n_img):
        m = (d["hid_t"] >= k * T) & (d["hid_t"] < (k + 1) * T)
        t, i = d["hid_t"][m] - k * T, d["hid_i"][m]
        first = np.full(n_hidden, np.nan)
        for tt, ii in zip(t[::-1], i[::-1]): first[ii] = tt     # last assignment = earliest spike
        lat.append(np.nanmean(first) if np.isfinite(first).any() else np.nan)
    return float(np.nanmean(lat))

for regime in ["fixed", "matched"]:
    for tau in TAUS:
        ws = BASE["w_scale"] if regime == "fixed" else BASE["w_scale"] * 20.0 / tau
        p = dict(BASE, tau_m=tau * ms, w_scale=ws)
        print(f"[{regime}] tau_m = {tau} ms, w_scale = {ws/mV:.3f} mV")
        if tau == 20:
            ctr, cte = counts_tr, counts_te
        else:
            ctr = simulate(Xtr, W0, p); cte = simulate(Xte, W0, p)
        r = train_readout(ctr.sum(2), ytr, cte.sum(2), yte, C=C_READOUT)
        d = demo_run(Xte[:20], W0, p, record_neurons=(0,))
        bin_s = float(p["t_enc"] / second) / p["n_bins"]
        psth = cte.sum(axis=(0, 1)) / (N_TEST * p["n_hidden"] * bin_s)
        res_tau[regime][tau] = dict(acc=r["acc"], f1=r["f1"], train_acc=r["train_acc"],
                                    mean_rate_hz=float(cte.sum((1, 2)).mean() / p["n_hidden"] / float(p["t_enc"] / second)),
                                    spikes_per_image=float(cte.sum((1, 2)).mean()),
                                    first_spike_ms=first_spike_latency(d, p["n_hidden"], 20, T), psth=psth.tolist(),
                                    silent_frac=float((ctr.sum((0, 2)) == 0).mean()))
        traces_tau[regime][tau] = (d["v_t"][d["v_t"] < T], d["v"][0][d["v_t"] < T], d["hid_t"][(d["hid_i"] == 0) & (d["hid_t"] < T)])
        print(f"    acc {r['acc']:.3f}  F1 {r['f1']:.3f}  rate {res_tau[regime][tau]['mean_rate_hz']:.1f} Hz  "
              f"first-spike {res_tau[regime][tau]['first_spike_ms']:.1f} ms")
RESULTS["tau_sweep"] = res_tau

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for k, regime in enumerate(["fixed", "matched"]):
    acc = [res_tau[regime][t]["acc"] for t in TAUS]
    spk = [res_tau[regime][t]["spikes_per_image"] for t in TAUS]
    lat = [res_tau[regime][t]["first_spike_ms"] for t in TAUS]
    ax[0].plot(TAUS, acc, "o-", color=PAL[k], label=f"{regime} weights")
    ax[1].plot(TAUS, spk, "o-", color=PAL[k], label=f"{regime} weights")
    ax[2].plot(TAUS, lat, "o-", color=PAL[k], label=f"{regime} weights")
from matplotlib.ticker import NullFormatter, NullLocator
for a in ax:
    a.set_xscale("log"); a.xaxis.set_minor_locator(NullLocator()); a.xaxis.set_minor_formatter(NullFormatter())
    a.set_xticks(TAUS); a.set_xticklabels(TAUS); a.set_xlabel(r"$\tau_m$ (ms)"); a.legend()
ax[0].set_ylabel("test accuracy"); ax[0].set_ylim(0, 1); ax[0].set_title(r"accuracy vs $\tau_m$")
ax[1].set_ylabel("hidden spikes per image"); ax[1].set_title("energy proxy: spikes per image")
ax[2].set_ylabel("mean first-spike latency (ms)"); ax[2].set_title("latency of first hidden spike\n(fixed weights, 5 ms: no spikes)")
savefig("fig_tau_sweep")

fig, ax = plt.subplots(2, 4, figsize=(15, 5.5), sharex=True)
for k, regime in enumerate(["fixed", "matched"]):
    for j, tau in enumerate(TAUS):
        t, v, sp = traces_tau[regime][tau]
        ax[k, j].plot(t, v, color=PAL[j], lw=0.9); ax[k, j].axhline(-50, color="#898781", lw=0.8, ls=":")
        ax[k, j].vlines(sp, -50, -42, color=PAL[j], lw=1)
        ax[k, j].set_title(rf"{regime}: $\tau_m$ = {tau} ms, {len(sp)} spikes"); ax[k, j].set_ylim(-72, -40)
    ax[k, 0].set_ylabel("v (mV), hidden neuron 0")
for a in ax[1]: a.set_xlabel("time (ms)")
savefig("fig_tau_vtraces")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
tb = (np.arange(10) + 0.5) * 10
for k, regime in enumerate(["fixed", "matched"]):
    for j, tau in enumerate(TAUS):
        ax[k].plot(tb, res_tau[regime][tau]["psth"], "o-", color=PAL[j], label=rf"$\tau_m$={tau} ms")
    ax[k].set_title(f"population PSTH ({regime} weights)"); ax[k].set_xlabel("time after image onset (ms)"); ax[k].legend()
ax[0].set_ylabel("population rate (Hz / neuron)")
savefig("fig_tau_psth")

### 4.1 Tweak 1 - analysis

*(filled in after the run)*

## 5. Tweak 2: sparse connectivity

Connection probability $p_{conn} \in \{1.0, 0.5, 0.2, 0.1, 0.05, 0.02\}$ (1.0 = dense, 0.05 = the value suggested in the
assignment). The EPSP scale is set to $w_{scale} = 0.25\,\mathrm{mV}/p_{conn}$ so that the **mean** input drive per hidden neuron is
the same for all $p_{conn}$ (the baseline is the $p_{conn}=0.2$ point). We also measure how *diverse* the hidden features are:
the mean pairwise correlation of hidden spike counts across test images.

In [ ]:
PCONN = [1.0, 0.5, 0.2, 0.1, 0.05, 0.02]
res_p, W_p = {}, {}
for pc in PCONN:
    p = dict(BASE, p_conn=pc, w_scale=0.25 * mV / pc)
    W, mask = make_weights(p, seed=RNG_SEED); W_p[pc] = W
    print(f"p_conn = {pc}: fan-in ~ {mask.sum(0).mean():.0f}, w_scale = {p['w_scale']/mV:.2f} mV")
    if pc == BASE["p_conn"]:
        ctr, cte = counts_tr, counts_te
    else:
        ctr = simulate(Xtr, W, p); cte = simulate(Xte, W, p)
    Ftr, Fte = ctr.sum(2), cte.sum(2)
    r = train_readout(Ftr, ytr, Fte, yte, C=C_READOUT)
    active = Fte.std(0) > 0
    corr = np.corrcoef(Fte[:, active].T); mean_corr = float((corr.sum() - active.sum()) / (active.sum() * (active.sum() - 1)))
    res_p[pc] = dict(acc=r["acc"], f1=r["f1"], train_acc=r["train_acc"], fan_in=float(mask.sum(0).mean()),
                     mean_rate_hz=float(Fte.mean() / float(p["t_enc"] / second)), spikes_per_image=float(Fte.sum(1).mean()),
                     silent_frac=float((Ftr.sum(0) == 0).mean()), feature_corr=mean_corr, rate_std_hz=float(Fte.mean(0).std() / float(p["t_enc"] / second)))
    print(f"    acc {r['acc']:.3f}  F1 {r['f1']:.3f}  rate {res_p[pc]['mean_rate_hz']:.1f} Hz  mean feature corr {mean_corr:.2f}")
RESULTS["conn_sweep"] = {str(k): v for k, v in res_p.items()}

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(PCONN, [res_p[c]["acc"] for c in PCONN], "o-", color=PAL[0]); ax[0].set_ylabel("test accuracy"); ax[0].set_ylim(0, 1)
ax[0].set_title("accuracy vs connection probability")
ax[1].plot(PCONN, [res_p[c]["feature_corr"] for c in PCONN], "o-", color=PAL[1]); ax[1].set_ylabel("mean pairwise feature correlation")
ax[1].set_title("feature redundancy")
ax[2].plot(PCONN, [res_p[c]["spikes_per_image"] for c in PCONN], "o-", color=PAL[2]); ax[2].set_ylabel("hidden spikes per image")
ax[2].set_title("energy proxy")
for a in ax:
    a.set_xscale("log"); a.xaxis.set_minor_locator(NullLocator()); a.xaxis.set_minor_formatter(NullFormatter())
    a.set_xticks(PCONN); a.set_xticklabels(PCONN); a.set_xlabel(r"$p_{conn}$"); a.invert_xaxis()
savefig("fig_conn_sweep")

fig, ax = plt.subplots(3, 8, figsize=(14, 5.6))
for row, pc in enumerate([1.0, 0.2, 0.05]):
    for j in range(8):
        ax[row, j].imshow(W_p[pc][:, j].reshape(28, 28), cmap="Blues", vmin=0); ax[row, j].axis("off")
    ax[row, 0].set_title(rf"$p_{{conn}}$ = {pc}", loc="left", fontsize=10)
fig.suptitle("input -> hidden weight 'filters' of the first 8 hidden neurons (random, frozen; no learning)", y=1.01)
savefig("fig_conn_filters")

### 5.1 Tweak 2 - analysis

*(filled in after the run)*

## 6. Tweak 3: homeostatic firing-rate regulation

Each hidden neuron gets an **adaptive threshold** $\theta_i$ and a low-pass estimate $r_i$ of its own firing rate:

$$\frac{dr_i}{dt} = -\frac{r_i}{\tau_r} \;(+\tfrac{1}{\tau_r}\text{ at each spike}), \qquad
\frac{d\theta_i}{dt} = \eta\,(r_i - r_{target}), \qquad \theta_i \in [V_{rest}+2\,\mathrm{mV},\, V_{th}+60\,\mathrm{mV}].$$

A neuron firing above target raises its threshold, one firing below target lowers it. We run this on a 500-image calibration stream, cycled so that each of the 25 parallel network copies adapts for
20 s of simulated time ($40\,\tau_r$; $\tau_r = 0.5$ s, $\eta = 0.5$ mV), then **freeze** the thresholds and re-run
train/test exactly as in the baseline. Targets: 10, 20 and 30 Hz.

The rate reached with *frozen* thresholds ends up below the target: a threshold that fluctuates during calibration yields a
higher mean rate than its frozen average, and the first 1 000 test images carry less ink than the calibration images.
We therefore always report the rate actually achieved, and compare homeostasis with the baseline **at equal spike budget**
(accuracy vs. spikes per image).

In [ ]:
res_h, theta_h = {}, {}
HOMEO_TARGETS = [10, 20, 30]
for rt in HOMEO_TARGETS:
    print(f"r_target = {rt} Hz")
    th, trace = calibrate_homeostasis(X_calib, W0, BASE, r_target=rt * Hz, tau_r=0.5 * second, eta=0.5 * mV)
    theta_h[rt] = (th, trace)
    ctr = simulate(Xtr, W0, BASE, theta_frozen=th); cte = simulate(Xte, W0, BASE, theta_frozen=th)
    Ftr, Fte = ctr.sum(2), cte.sum(2)
    r = train_readout(Ftr, ytr, Fte, yte, C=C_READOUT)
    rates = Fte.mean(0) / float(BASE["t_enc"] / second)
    res_h[rt] = dict(acc=r["acc"], f1=r["f1"], train_acc=r["train_acc"], mean_rate_hz=float(rates.mean()), rate_std_hz=float(rates.std()),
                     spikes_per_image=float(Fte.sum(1).mean()), theta_mean_mV=float(np.mean(th / mV)), theta_std_mV=float(np.std(th / mV)),
                     rates=rates.tolist())
    print(f"    acc {r['acc']:.3f}  F1 {r['f1']:.3f}  rate {rates.mean():.1f} +- {rates.std():.1f} Hz")
RESULTS["homeostasis"] = {str(k): {kk: vv for kk, vv in v.items() if kk != "rates"} for k, v in res_h.items()}

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
th, trace = theta_h[20]
ax[0].plot(np.arange(1, trace.shape[0] + 1) * float(BASE["t_enc"] / second), trace[:, :12], lw=0.9)
ax[0].set_xlabel("simulated calibration time (s)"); ax[0].set_ylabel(r"$\theta$ (mV)"); ax[0].set_title("adaptive thresholds of 12 neurons (target 20 Hz)")
for k, rt in enumerate(HOMEO_TARGETS):
    ax[1].hist(theta_h[rt][0] / mV, bins=25, alpha=0.7, color=PAL[k], label=f"target {rt} Hz")
ax[1].axvline(-50, color="#898781", ls=":")
ax[1].set_xlabel(r"frozen threshold $\theta$ (mV)"); ax[1].set_ylabel("# neurons"); ax[1].legend(); ax[1].set_title("threshold distribution after calibration")
ax[2].hist(rate_hz, bins=25, alpha=0.7, color="#898781", label=f"baseline (acc {base['acc']:.3f})")
for k, rt in enumerate(HOMEO_TARGETS):
    ax[2].hist(res_h[rt]["rates"], bins=25, alpha=0.7, color=PAL[k], label=f"target {rt} Hz (acc {res_h[rt]['acc']:.3f})")
ax[2].set_xlabel("mean rate per hidden neuron (Hz)"); ax[2].set_ylabel("# neurons"); ax[2].legend(); ax[2].set_title("rate distribution with frozen thresholds")
savefig("fig_homeostasis")

# equal-spike-budget comparison: homeostasis vs. networks whose spike count was changed via tau_m (matched weights)
fig, ax = plt.subplots(figsize=(6.5, 4.2))
pts = sorted((res_tau["matched"][t]["spikes_per_image"], res_tau["matched"][t]["acc"]) for t in TAUS)
ax.plot(*zip(*pts), "o-", color=PAL[1], label=r"no homeostasis ($\tau_m$ sweep, matched weights)")
ax.plot(RESULTS["baseline"]["spikes_per_image"], RESULTS["baseline"]["acc"], "s", ms=9, color="#52514e", label="baseline")
ax.plot([res_h[t]["spikes_per_image"] for t in HOMEO_TARGETS], [res_h[t]["acc"] for t in HOMEO_TARGETS], "D-",
        color=PAL[0], ms=7, label="homeostasis (10 / 20 / 30 Hz targets)")
ax.set_xscale("log"); ax.set_xlabel("hidden spikes per image (log scale)"); ax.set_ylabel("test accuracy")
ax.legend(fontsize=8); ax.set_title("accuracy at equal spike budget")
savefig("fig_homeostasis_budget")

### 6.1 Tweak 3 - analysis

*(filled in after the run)*

## 7. Summary of all experiments

In [ ]:
rows = [("baseline (tau_m=20 ms, p=0.2)", RESULTS["baseline"])]
rows += [(f"tau_m={t} ms, fixed w", res_tau["fixed"][t]) for t in TAUS if t != 20]
rows += [(f"tau_m={t} ms, matched w", res_tau["matched"][t]) for t in TAUS if t != 20]
rows += [(f"p_conn={c}", res_p[c]) for c in PCONN if c != 0.2]
rows += [(f"homeostasis target {t} Hz", res_h[t]) for t in HOMEO_TARGETS]
print(f"{'configuration':34s} {'acc':>6s} {'F1':>6s} {'rate Hz':>8s} {'spk/img':>8s}")
for name, r in rows:
    print(f"{name:34s} {r['acc']:6.3f} {r['f1']:6.3f} {r['mean_rate_hz']:8.1f} {r['spikes_per_image']:8.0f}")
with open("results/results.json", "w") as f:
    json.dump(RESULTS, f, indent=1)

## 8. Conclusions

*(filled in after the run)*